# FDS Challenge: Starter Notebook

This notebook will guide you through the first steps of the competition. Our goal here is to show you how to:

1.  Load the `train.jsonl` and `test.jsonl` files from the competition data.
2.  Create a very simple set of features from the data.
3.  Train a basic model.
4.  Generate a `submission.csv` file in the correct format.
5.  Submit your results.

Let's get started!

### 1. Loading and Inspecting the Data

When you create a notebook within a Kaggle competition, the competition's data is automatically attached and available in the `../input/` directory.

The dataset is in a `.jsonl` format, which means each line is a separate JSON object. This is great because we can process it one line at a time without needing to load the entire large file into memory.

Let's write a simple loop to load the training data and inspect the first battle.

In [17]:
import json
import pandas as pd
from pathlib import Path

#Couldn't add the data to the git so had to make local path, below allows the local and the kaggle path to be used interchangeably
KAGGLE_DATA = Path(
    "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026"
)
if KAGGLE_DATA.exists():
    DATA_DIR = KAGGLE_DATA
else:
    DATA_DIR = Path("data")
train_file_path = DATA_DIR / "train.jsonl"
test_file_path = DATA_DIR / "test.jsonl"
sample_submission_path = DATA_DIR / "sample_submission.csv"
type_chart_path = DATA_DIR / "type_chart_gen2.csv"

type_chart_df = pd.read_csv(type_chart_path)

type_chart = {}

for _, row in type_chart_df.iterrows():
    attacking_type = str(row['attacking_type']).lower()
    defending_type = str(row['defending_type']).lower()
    multiplier = float(row['multiplier'])

    type_chart[(attacking_type, defending_type)] = multiplier

print(f"Loaded type chart with {len(type_chart)} effectiveness multipliers.")

train_data = []
test_data = []

print(f"Loading data from '{test_file_path}'...")

with open(test_file_path, 'r') as f:
    for line in f:
        test_data.append(json.loads(line))

print(f"Successfully loaded {len(test_data)} test battles.")

# Read the file line by line
print(f"Loading data from '{train_file_path}'...")
try:
    with open(train_file_path, 'r') as f:
        for line in f:
            # json.loads() parses one line (one JSON object) into a Python dictionary
            train_data.append(json.loads(line))

    print(f"Successfully loaded {len(train_data)} battles.")

    # Let's inspect the first battle to see its structure
    print("\n--- Structure of the first train battle: ---")
    if train_data:
        first_battle = train_data[0]
        
        # To keep the output clean, we can create a copy and truncate the timeline
        battle_for_display = first_battle.copy()
        battle_for_display['battle_timeline'] = battle_for_display.get('battle_timeline', [])[:2] # Show first 2 turns
        
        # Use json.dumps for pretty-printing the dictionary
        print(json.dumps(battle_for_display, indent=4))
        if len(first_battle.get('battle_timeline', [])) > 3:
            print("    ...")
            print("    (battle_timeline has been truncated for display)")


except FileNotFoundError:
    print(f"ERROR: Could not find the training file at '{train_file_path}'.")
    print("Please make sure you have added the competition data to this notebook.")

Loaded type chart with 110 effectiveness multipliers.
Loading data from 'data/test.jsonl'...
Successfully loaded 5000 test battles.
Loading data from 'data/train.jsonl'...
Successfully loaded 10000 battles.

--- Structure of the first train battle: ---
{
    "battle_id": 0,
    "p1_team_details": [
        {
            "name": "tyranitar",
            "level": 100,
            "types": [
                "dark",
                "rock"
            ],
            "base_hp": 100,
            "base_atk": 134,
            "base_def": 110,
            "base_spa": 95,
            "base_spd": 100,
            "base_spe": 61,
            "item": "leftovers"
        },
        {
            "name": "typhlosion",
            "level": 100,
            "types": [
                "fire",
                "notype"
            ],
            "base_hp": 78,
            "base_atk": 84,
            "base_def": 78,
            "base_spa": 109,
            "base_spd": 85,
            "base_spe": 100,
      

### 2. Basic Feature Engineering

A successful model will likely require creating many complex features. For this starter notebook, however, we will create a very simple feature set based **only on the initial team stats**. This will be enough to train a model and generate a submission file.

It's up to you to engineer more powerful features!

In [18]:
# Build a mapping of Pokémon species to their elemental types
def build_species_types_map(data):
    species_types = {}

    for battle in data:
        p1_team = battle.get('p1_team_details', [])

        for pokemon in p1_team:
            name = pokemon['name']
            types = pokemon.get('types', [])

            if name not in species_types:
                species_types[name] = types

        p2_lead = battle.get('p2_lead_details')

        if p2_lead:
            name = p2_lead['name']
            types = p2_lead.get('types', [])

            if name not in species_types:
                species_types[name] = types

    return species_types


species_map = build_species_types_map(train_data)

print(f"> Mapped {len(species_map)} Pokémon species with their elemental types.")


def get_type_mult(attacking_types, defending_types):
    multiplier = 1.0

    for attack_type in attacking_types:
        if not attack_type or attack_type == 'notype':
            continue

        for defend_type in defending_types:
            if not defend_type or defend_type == 'notype':
                continue

            attack_type = attack_type.lower()
            defend_type = defend_type.lower()

            type_multiplier = type_chart.get(
                (attack_type, defend_type),
                1.0
            )

            multiplier = multiplier * type_multiplier

    return multiplier

> Mapped 239 Pokémon species with their elemental types.


In [19]:
from tqdm.auto import tqdm
import numpy as np

HEAL_MOVES = {
    'rest', 'recover', 'softboiled',
    'synthesis', 'moonlight', 'morningsun'
}
SETUP_MOVES = {
    'curse', 'swordsdance', 'bellydrum',
    'agility', 'growth'
}

def create_simple_features(data):
    feature_rows = []
    for battle in tqdm(data, desc="Extracting features"):
        features = {
            'battle_id': battle['battle_id']
        }
        if 'player_won' in battle:
            features['player_won'] = int(battle['player_won'])

        # P1 team base stats

        p1_team = battle.get('p1_team_details', [])
        p1_names = {pokemon['name'] for pokemon in p1_team}

        stats = ['hp', 'atk', 'def', 'spa', 'spd', 'spe']
        for stat in stats:
            values = []
            for pokemon in p1_team:
                values.append(
                    pokemon.get(f'base_{stat}', 0)
                )
            features[f'p1_mean_{stat}'] = float(np.mean(values))
            features[f'p1_std_{stat}'] = float(np.std(values))


        # Lead matchup

        p2_lead = battle.get('p2_lead_details', {})
        if p1_team and p2_lead:
            p1_lead = p1_team[0]
            p1_speed = p1_lead.get('base_spe', 0)
            p2_speed = p2_lead.get('base_spe', 0)
            features['lead_spe_diff'] = float(
                p1_speed - p2_speed
            )
            p1_type_effectiveness = get_type_mult(
                p1_lead.get('types', []),
                p2_lead.get('types', [])
            )
            p2_type_effectiveness = get_type_mult(
                p2_lead.get('types', []),
                p1_lead.get('types', [])
            )
            features['lead_type_adv'] = float(
                p1_type_effectiveness - p2_type_effectiveness
            )

        # Timeline tracking

        p1_hp = {
            name: 1.0
            for name in p1_names
        }
        p2_hp = {}
        actions = {
            'p1_sw': 0,
            'p2_sw': 0,
            'p1_spikes': 0,
            'p2_spikes': 0,
            'p1_heals': 0,
            'p2_heals': 0,
            'p1_setups': 0,
            'p2_setups': 0
        }
        p1_hp_turn20 = 1.0
        p2_hp_turn20 = 1.0
        timeline = battle.get('battle_timeline', [])
        for turn in timeline:
            p1_state = turn.get('p1_pokemon_state', {})
            p2_state = turn.get('p2_pokemon_state', {})

            p1_move = turn.get('p1_move_details')
            p2_move = turn.get('p2_move_details')

            # Update HP

            p1_name = p1_state.get('name')
            if p1_name:
                p1_hp[p1_name] = p1_state.get('hp_pct', 1.0)

                if turn.get('turn') == 20:
                    p1_hp_turn20 = p1_hp[p1_name]
            p2_name = p2_state.get('name')
            if p2_name:
                p2_hp[p2_name] = p2_state.get('hp_pct', 1.0)

                if turn.get('turn') == 20:
                    p2_hp_turn20 = p2_hp[p2_name]


            # Count actions

            for move, player in [
                (p1_move, 'p1'),
                (p2_move, 'p2')
            ]:

                if move is None:
                    actions[f'{player}_sw'] += 1
                    continue

                move_name = move.get('name', '')

                if move_name == 'spikes':
                    actions[f'{player}_spikes'] += 1

                elif move_name in HEAL_MOVES:
                    actions[f'{player}_heals'] += 1

                elif move_name in SETUP_MOVES:
                    actions[f'{player}_setups'] += 1


        # Overall team state

        p1_total_hp = sum(p1_hp.values())
        unseen_p2 = max(0, 6 - len(p2_hp))
        p2_total_hp = (
            sum(p2_hp.values())
            + unseen_p2
        )
        p1_alive = sum(
            hp > 0
            for hp in p1_hp.values()
        )

        p2_alive = (
            sum(hp > 0 for hp in p2_hp.values())
            + unseen_p2
        )

        features['p1_total_hp'] = float(p1_total_hp)
        features['p2_total_hp'] = float(p2_total_hp)

        features['total_hp_diff'] = float(
            p1_total_hp - p2_total_hp
        )

        features['alive_diff'] = float(
            p1_alive - p2_alive
        )


        # Final active Pokémon state

        if timeline:

            final_turn = timeline[-1]

            p1_final = final_turn.get(
                'p1_pokemon_state', {}
            )

            p2_final = final_turn.get(
                'p2_pokemon_state', {}
            )

            p1_final_hp = p1_final.get('hp_pct', 0.0)
            p2_final_hp = p2_final.get('hp_pct', 0.0)

            features['act_hp_p1'] = float(p1_final_hp)
            features['act_hp_p2'] = float(p2_final_hp)

            features['act_hp_diff'] = float(
                p1_final_hp - p2_final_hp
            )


            p1_final_types = species_map.get(
                p1_final.get('name', ''),
                []
            )

            p2_final_types = species_map.get(
                p2_final.get('name', ''),
                []
            )

            p1_final_effectiveness = get_type_mult(
                p1_final_types,
                p2_final_types
            )

            p2_final_effectiveness = get_type_mult(
                p2_final_types,
                p1_final_types
            )

            features['act_type_adv'] = float(
                p1_final_effectiveness
                - p2_final_effectiveness
            )


            p1_has_status = p1_final.get(
                'status'
            ) not in (None, 'nostatus')

            p2_has_status = p2_final.get(
                'status'
            ) not in (None, 'nostatus')

            features['act_status_diff'] = float(
                p2_has_status - p1_has_status
            )


        # Momentum and tactical features

        features['net_hp_trend'] = float(
            (p1_total_hp - p1_hp_turn20 * 6.0)
            -
            (p2_total_hp - p2_hp_turn20 * 6.0)
        )

        features['spikes_diff'] = float(
            actions['p1_spikes']
            - actions['p2_spikes']
        )

        features['switches_diff'] = float(
            actions['p1_sw']
            - actions['p2_sw']
        )

        features['heals_diff'] = float(
            actions['p1_heals']
            - actions['p2_heals']
        )

        features['setups_diff'] = float(
            actions['p1_setups']
            - actions['p2_setups']
        )


        p1_snorlax_alive = (
            'snorlax' in p1_names
            and p1_hp.get('snorlax', 1.0) > 0
        )

        p2_snorlax_alive = (
            'snorlax' in p2_hp
            and p2_hp.get('snorlax', 1.0) > 0
        )
        features['snorlax_diff'] = float(
            p1_snorlax_alive - p2_snorlax_alive
        )
        feature_rows.append(features)


    return pd.DataFrame(feature_rows)


print("Extracting features from training data...")
train_df = create_simple_features(train_data)
test_df = create_simple_features(test_data)
print(
    f"Features extracted: "
    f"{train_df.shape[0]} rows, "
    f"{train_df.shape[1]} columns."
)
train_df.head()

Extracting features from training data...


Extracting features:   0%|          | 0/10000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]

Features extracted: 10000 rows, 31 columns.


,battle_id,player_won,p1_mean_hp,p1_std_hp,p1_mean_atk,p1_std_atk,p1_mean_def,p1_std_def,p1_mean_spa,p1_std_spa,...,act_hp_p2,act_hp_diff,act_type_adv,act_status_diff,net_hp_trend,spikes_diff,switches_diff,heals_diff,setups_diff,snorlax_diff
0,0,0,83.000000,14.719601,88.000000,27.682726,91.333333,28.235124,94.833333,28.927592,...,0.850000,0.150000,3.75,-1.0,5.010000,-1.0,0.0,1.0,-1.0,-1.0
1,1,0,91.666667,33.499585,83.333333,15.184056,99.166667,33.839901,75.833333,24.051449,...,0.740000,-0.740000,-0.25,-1.0,0.240000,1.0,-2.0,0.0,0.0,0.0
2,2,0,95.000000,33.040379,113.166667,18.215531,97.500000,39.449335,77.500000,24.109127,...,0.937337,-0.749347,0.00,1.0,-3.427012,2.0,1.0,0.0,4.0,-1.0
3,3,0,119.166667,69.846061,75.000000,32.145503,98.333333,54.211520,75.000000,26.299556,...,0.950000,-0.410000,0.00,0.0,0.770000,0.0,0.0,7.0,1.0,0.0
4,4,0,105.166667,30.140873,89.500000,13.400871,95.333333,47.737709,94.166667,27.751376,...,0.000000,0.000000,-0.50,0.0,0.540000,0.0,-1.0,-1.0,-3.0,0.0


### 3. Training a Baseline Model

Now that we have some features, let's train a simple `LogisticRegression` model. This will give us a starting point for our predictions.

In [20]:
features = [col for col in train_df.columns if col not in ['battle_id', 'player_won']]

X = train_df[features]
y = train_df['player_won']

X_test = test_df[features]

print("Number of features:", len(features))

Number of features: 29


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)
scores = cross_val_score(
    log_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("Logistic Regression accuracies:", scores)
print(f"Mean validation accuracy: {scores.mean()*100:.2f}%")

log_model.fit(X, y)
log_predictions = log_model.predict(X_test)
# submission_logistic = pd.read_csv(sample_submission_path)
# submission_logistic['player_won'] = log_predictions
# submission_logistic.to_csv(
#     "submissions/submission_logistic_regression.csv",
#     index=False
# )
# print("Logistic Regression submission created.")

Logistic Regression accuracies: [0.665  0.664  0.667  0.6625 0.6605]
Mean validation accuracy: 66.38%


In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.04,
    max_leaf_nodes=35,
    min_samples_leaf=20,
    l2_regularization=2.0,
    random_state=42
)
hist_scores = cross_val_score(
    hist_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("HistGradientBoosting accuracies:", hist_scores)
print(f"Mean validation accuracy: {hist_scores.mean()*100:.2f}%")

hist_model.fit(X, y)
hist_predictions = hist_model.predict(X_test)

# submission_hist = pd.read_csv(sample_submission_path)
# submission_hist['player_won'] = hist_predictions
# submission_hist.to_csv(
#     "submissions/submission_hist_gradient_boosting.csv",
#     index=False
# )
# print("HistGradientBoosting submission created.")

HistGradientBoosting accuracies: [0.662  0.678  0.671  0.6705 0.6615]
Mean validation accuracy: 66.86%


In [23]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.04,
    num_leaves=31,
    min_child_samples=25,
    reg_alpha=1.0,
    reg_lambda=2.0,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm_scores = cross_val_score(
    lgbm_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)
print("LightGBM fold accuracies:", lgbm_scores)
print(f"Mean validation accuracy: {lgbm_scores.mean()*100:.2f}%")

lgbm_model.fit(X, y)
lgbm_predictions = lgbm_model.predict(X_test)

# submission_lgbm = pd.read_csv(sample_submission_path)
# submission_lgbm['player_won'] = lgbm_predictions
# submission_lgbm.to_csv(
#     "submissions/submission_lightgbm.csv",
#     index=False
# )
# print("LightGBM submission created.")

LightGBM fold accuracies: [0.669  0.672  0.6665 0.6615 0.6695]
Mean validation accuracy: 66.77%


In [24]:
%pip install xgboost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.04,
    max_depth=5,
    min_child_weight=20,
    reg_alpha=1.0,
    reg_lambda=2.0,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb_scores = cross_val_score(
    xgb_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)
print("XGBoost fold accuracies:", xgb_scores)
print(f"Mean validation accuracy: {xgb_scores.mean()*100:.2f}%")
xgb_model.fit(X, y)
xgb_predictions = xgb_model.predict(X_test)

# submission_xgb = pd.read_csv(sample_submission_path)
# submission_xgb['player_won'] = xgb_predictions
# submission_xgb.to_csv(
#     "submissions/submission_xgboost.csv",
#     index=False
# )
# print("XGBoost submission created.")

Note: you may need to restart the kernel to use updated packages.
XGBoost fold accuracies: [0.664  0.6735 0.669  0.6655 0.664 ]
Mean validation accuracy: 66.72%


In [25]:
from sklearn.model_selection import StratifiedKFold

models = {
    'HistGB': hist_model,
    'LightGBM': lgbm_model,
    'XGBoost': xgb_model
}
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
oof_predictions = {
    name: np.zeros(len(y))
    for name in models
}
for train_index, val_index in skf.split(X, y):
    X_train = X.iloc[train_index]
    X_val = X.iloc[val_index]
    y_train = y.iloc[train_index]

    for name, model in models.items():
        model.fit(X_train, y_train)
        oof_predictions[name][val_index] = model.predict_proba(X_val)[:, 1]

ensemble_probabilities = np.mean(
    [
        oof_predictions['HistGB'],
        oof_predictions['LightGBM'],
        oof_predictions['XGBoost']
    ],
    axis=0
)

ensemble_predictions = (ensemble_probabilities > 0.5).astype(int)
ensemble_accuracy = accuracy_score(
    y,
    ensemble_predictions
)
print(f"Ensemble validation accuracy: {ensemble_accuracy * 100:.2f}%")
for model in models.values():
    model.fit(X, y)
test_probabilities = np.mean(
    [
        models['HistGB'].predict_proba(X_test)[:, 1],
        models['LightGBM'].predict_proba(X_test)[:, 1],
        models['XGBoost'].predict_proba(X_test)[:, 1]
    ],
    axis=0
)
test_predictions = (test_probabilities > 0.5).astype(int)

submission_ensemble = pd.read_csv(sample_submission_path)
submission_ensemble['player_won'] = test_predictions

submission_ensemble.to_csv(
    "submissions/submission_ensemble.csv",
    index=False
)

print("Ensemble submission created.")

Ensemble validation accuracy: 66.78%
Ensemble submission created.


In [26]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from scipy.optimize import minimize

models = {
    'HistGB': hist_model,
    'LightGBM': lgbm_model,
    'XGBoost': xgb_model
}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions = {
    name: np.zeros(len(y))
    for name in models
}

for train_index, val_index in skf.split(X, y):
    X_train = X.iloc[train_index]
    X_val = X.iloc[val_index]
    y_train = y.iloc[train_index]

    for name, model in models.items():
        model.fit(X_train, y_train)
        oof_predictions[name][val_index] = model.predict_proba(X_val)[:, 1]

def ensemble_loss(weights):
    weights = weights / np.sum(weights)

    probabilities = (
        weights[0] * oof_predictions['HistGB']
        + weights[1] * oof_predictions['LightGBM']
        + weights[2] * oof_predictions['XGBoost']
    )

    return -accuracy_score(
        y,
        probabilities > 0.5
    )

result = minimize(
    ensemble_loss,
    [1/3, 1/3, 1/3],
    bounds=[(0, 1), (0, 1), (0, 1)]
)

weights = result.x / np.sum(result.x)

ensemble_probabilities = (
    weights[0] * oof_predictions['HistGB']
    + weights[1] * oof_predictions['LightGBM']
    + weights[2] * oof_predictions['XGBoost']
)

best_threshold = 0.5
best_accuracy = accuracy_score(
    y,
    ensemble_probabilities > best_threshold
)

for threshold in np.arange(0.46, 0.541, 0.001):
    accuracy = accuracy_score(
        y,
        ensemble_probabilities > threshold
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = threshold

print(f"HistGB weight: {weights[0]:.3f}")
print(f"LightGBM weight: {weights[1]:.3f}")
print(f"XGBoost weight: {weights[2]:.3f}")
print(f"Best threshold: {best_threshold:.3f}")
print(f"Weighted ensemble validation accuracy: {best_accuracy * 100:.2f}%")

for model in models.values():
    model.fit(X, y)

test_probabilities = (
    weights[0] * models['HistGB'].predict_proba(X_test)[:, 1]
    + weights[1] * models['LightGBM'].predict_proba(X_test)[:, 1]
    + weights[2] * models['XGBoost'].predict_proba(X_test)[:, 1]
)

test_predictions = (
    test_probabilities > best_threshold
).astype(int)

submission_weighted = pd.read_csv(sample_submission_path)
submission_weighted['player_won'] = test_predictions

submission_weighted.to_csv(
    "submissions/submission_weighted_ensemble.csv",
    index=False
)

print("Weighted ensemble submission created.")

HistGB weight: 0.333
LightGBM weight: 0.333
XGBoost weight: 0.333
Best threshold: 0.531
Weighted ensemble validation accuracy: 67.09%
Weighted ensemble submission created.


In [27]:
best_accuracy = 0
best_weights = None
best_threshold = 0.5

for hist_weight in np.arange(0, 1.01, 0.05):

    for lgbm_weight in np.arange(0, 1.01 - hist_weight, 0.05):

        xgb_weight = 1 - hist_weight - lgbm_weight

        probabilities = (
            hist_weight * oof_predictions['HistGB']
            + lgbm_weight * oof_predictions['LightGBM']
            + xgb_weight * oof_predictions['XGBoost']
        )

        for threshold in np.arange(0.46, 0.541, 0.001):

            predictions = (
                probabilities > threshold
            ).astype(int)

            accuracy = accuracy_score(
                y,
                predictions
            )

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_weights = (
                    hist_weight,
                    lgbm_weight,
                    xgb_weight
                )
                best_threshold = threshold


print("Best weights:")
print(f"HistGB: {best_weights[0]:.2f}")
print(f"LightGBM: {best_weights[1]:.2f}")
print(f"XGBoost: {best_weights[2]:.2f}")

print(f"Best threshold: {best_threshold:.3f}")
print(f"Best validation accuracy: {best_accuracy * 100:.2f}%")

Best weights:
HistGB: 0.25
LightGBM: 0.30
XGBoost: 0.45
Best threshold: 0.514
Best validation accuracy: 67.17%


In [28]:
for model in models.values():
    model.fit(X, y)

hist_prob = models['HistGB'].predict_proba(X_test)[:, 1]
lgbm_prob = models['LightGBM'].predict_proba(X_test)[:, 1]
xgb_prob = models['XGBoost'].predict_proba(X_test)[:, 1]

test_probabilities = (
    best_weights[0] * hist_prob
    + best_weights[1] * lgbm_prob
    + best_weights[2] * xgb_prob
)

test_predictions = (
    test_probabilities > best_threshold
).astype(int)

submission_weighted = pd.read_csv(sample_submission_path)
submission_weighted['player_won'] = test_predictions

submission_weighted.to_csv(
    "submissions/submission_weighted_ensemble.csv",
    index=False
)

print("Weighted ensemble submission created.")

Weighted ensemble submission created.


### 4. Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [29]:
# # Make predictions on the test data
# print("Generating predictions on the test set...")
# test_predictions = model.predict(X_test)

# # Use the provided sample file as the submission template.
# sample_submission = pd.read_csv(sample_submission_path)
# expected_columns = ['battle_id', 'player_won']
# if list(sample_submission.columns) != expected_columns:
#     raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

# # Match predictions to the template by battle_id, preserving its row order.
# predictions_by_id = pd.Series(test_predictions, index=test_df['battle_id'])
# submission_df = sample_submission.copy()
# submission_df['player_won'] = submission_df['battle_id'].map(predictions_by_id)
# if submission_df['player_won'].isna().any():
#     raise ValueError("Some sample submission battle_id values have no matching test prediction.")
# submission_df['player_won'] = submission_df['player_won'].astype(int)

# submission_df.to_csv('submission.csv', index=False)

# # Once you create the submission file, you will find it in /kaggle/working
# print("\n'submission.csv' file created successfully!")
# display(submission_df.head())

### 5. Submitting Your Results

Once you have generated your `submission.csv` file, there are two primary ways to submit it to the competition.

---

#### Method A: Submitting Directly from the Notebook (Recommended)

This is the standard method for code competitions. It ensures that your submission is linked to the code that produced it, which is crucial for reproducibility.

1.  **Save Your Work:** Click the **"Save Version"** button in the top-right corner of the notebook editor.
2.  **Run the Notebook:** In the pop-up window, select **"Save & Run All (Commit)"** and then click the **"Save"** button. This will run your entire notebook from top to bottom and save the output, including your `submission.csv` file.
3.  **Go to the Viewer:** Once the save process is complete, navigate to the notebook viewer page. 
4.  **Submit to Competition:** In the viewer, find the **"Submit to Competition"** section. This is usually located in the header of the output section or in the vertical "..." menu on the right side of the page. Clicking the **Submit** button this will submit your generated `submission.csv` file.

After submitting, you will see your score in the **"Submit to Competition"** section or in the [Public Leaderboard](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?).

---

#### Method B: Manual Upload

You can also generate your predictions and submission file using any environment you prefer (this notebook, Google Colab, or your local machine).

1.  **Generate the `submission.csv` file** using your model.
2.  **Download the file** to your computer.
3.  **Navigate to the [Leaderboard Page](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?)** and click on the **"Submit Predictions"** button.
4.  **Upload Your File:** Drag and drop or select your `submission.csv` file to upload it.

This method is quick, but keep in mind that for the final evaluation, you might be required to provide the code that generated your submission.

Good luck!